In [3]:
##This script automates the extraction, conversion, and upload of images from Figma to Azure Blob Storage

import os
import requests
from PIL import Image
import io
import base64
import asyncio
from typing import Literal
from dotenv import load_dotenv
from urllib.parse import urlparse
from itertools import islice

from azure.storage.blob import ContentSettings
from azure.storage.blob.aio import ContainerClient
from azure.core.pipeline.transport import AioHttpTransport

# === Load environment ===
load_dotenv(".env")
FIGMA_TOKEN = os.getenv("FIGMA_TOKEN")
FILE_ID_1 = os.getenv("FIGMA_DOCUMENT_ID1")
FILE_ID_2 = os.getenv("FIGMA_DOCUMENT_ID2")
SAS_URL = os.getenv("SAS_URL")

# === SSL override toggle ===
USE_INSECURE_SSL = False

# === Sanity check ===
missing = [var for var in ["FIGMA_TOKEN", "FILE_ID_1", "FILE_ID_2", "SAS_URL"] if not globals()[var]]
if missing:
    raise EnvironmentError(f"Missing required environment variables: {', '.join(missing)}")

def convert_to_thumbnail(base64_str: str, resize_to=1080, target_format="webp", lossless=False, quality=95) -> bytes:
    """
    Convert base64 image string to resized WebP thumbnail (1080p max).
    """
    image_bytes = base64.b64decode(base64_str)
    with Image.open(io.BytesIO(image_bytes)) as img:
        img = img.convert("RGB")
        if img.width > resize_to or img.height > resize_to:
            img.thumbnail((resize_to, resize_to), Image.Resampling.LANCZOS)
        output = io.BytesIO()
        if lossless:
            img.save(output, format=target_format.upper(), lossless=True)
        else:
            img.save(output, format=target_format.upper(), quality=quality)
        return output.getvalue()



# === Figma Image ID Extraction ===
def get_image_ids(file_id: str, token: str, pages: list[str]) -> dict[str, list[str]]:
    url = f"https://api.figma.com/v1/files/{file_id}"
    headers = {"X-Figma-Token": token}
    res = requests.get(url, headers=headers)
    res.raise_for_status()
    data = res.json()

    results = {}

    def collect_image_fills(node, found):
        if "fills" in node:
            for fill in node["fills"]:
                if fill.get("type") == "IMAGE":
                    found.append(node["id"])
                    break
        for child in node.get("children", []):
            collect_image_fills(child, found)

    for page in data["document"]["children"]:
        if page["name"] in pages:
            found = []
            collect_image_fills(page, found)
            results[page["name"]] = found

    return results


# === Upload to Azure ===
async def upload_image_to_azure(entity_id: str, page_name: str, image_data: bytes, overwrite: bool = False) -> str:
    folder_path = f"{page_name.replace(' ', '_')}"
    blob_name = f"{folder_path}/{entity_id}.webp"

    if USE_INSECURE_SSL:
        import ssl
        ssl_context = ssl.create_default_context()
        ssl_context.check_hostname = False
        ssl_context.verify_mode = ssl.CERT_NONE
        transport = AioHttpTransport(ssl_context=ssl_context)
        client = ContainerClient.from_container_url(SAS_URL, transport=transport)
    else:
        client = ContainerClient.from_container_url(SAS_URL)

    async with client:
        blob_client = client.get_blob_client(blob_name)
        blob_exists = await blob_client.exists()

        if blob_exists and not overwrite:
            print(f"⏭️ Skipped {entity_id} (already exists)")
            return blob_client.url.split("?")[0]

        await blob_client.upload_blob(
            data=image_data,
            overwrite=True,
            blob_type="BlockBlob",
            content_settings=ContentSettings(content_type="image/webp"),
        )
        return blob_client.url.split("?")[0]



# === Process Each Page ===
async def upload_figma_images_to_azure(
    file_id: str,
    page_name: str,
    node_ids: list[str],
    overwrite: bool = False
):
    if not node_ids:
        print(f"⚠️ No images found in page: {page_name}")
        return

    headers = {"X-Figma-Token": FIGMA_TOKEN}

    
    def chunked(iterable, size):
        it = iter(iterable)
        while True:
            chunk = list(islice(it, size))
            if not chunk:
                break
            yield chunk
    
    image_urls = {}
    
    for batch in chunked(node_ids, 20):
        try:
            res = requests.get(
                f"https://api.figma.com/v1/images/{file_id}",
                params={"ids": ",".join(batch), "format": "png"},
                headers=headers,
                timeout=60,
            )
            res.raise_for_status()
            image_urls.update(res.json().get("images", {}))
        except Exception as e:
            print(f"❌ Failed to fetch batch from Figma: {e}")


    for node_id in reversed(node_ids):
        url = image_urls.get(node_id)
        try:
            img_res = requests.get(url)
            img_res.raise_for_status()
            binary = img_res.content
            base64_img = base64.b64encode(binary).decode("utf-8")

            webp_data = convert_to_thumbnail(base64_img, resize_to=1080, target_format="webp", lossless=False, quality=95)
            clean_id = node_id.replace(":", "_")

            blob_url = await upload_image_to_azure(clean_id, "Modules/"+page_name, webp_data, overwrite=overwrite)
            print(f"✅ Uploaded {clean_id} → {blob_url}")

        except Exception as e:
            print(f"❌ Failed to process {node_id}: {e}")


pages_1=["Module 1","Module 2", "Module 3","Module 4"]
pages_2=["Module 5", "Module 6","Module 7","Module 8"]

# === Pipeline Runner ===
async def run_pipeline(overwrite):

    images_1 = get_image_ids(FILE_ID_1, FIGMA_TOKEN, pages_1)
    images_2 = get_image_ids(FILE_ID_2, FIGMA_TOKEN, pages_2)

    for page, ids in images_1.items():
        print(f"\n📦 Processing {page} from FILE_ID_1...")
        await upload_figma_images_to_azure(FILE_ID_1, page, ids, overwrite=overwrite)

    for page, ids in images_2.items():
        print(f"\n📦 Processing {page} from FILE_ID_2...")
        await upload_figma_images_to_azure(FILE_ID_2, page, ids, overwrite=overwrite)





In [ ]:
 #=== Run in notebook or script ===
await run_pipeline(overwrite=False)


📦 Processing Module 1 from FILE_ID_1...


In [4]:
##export specifically for lessons covers

import json
import os
import re
import base64
import requests
from itertools import islice
from azure.storage.blob.aio import ContainerClient

FIGMA_TOKEN = os.environ["FIGMA_TOKEN"]
SAS_URL = os.environ["SAS_URL"]

def get_container_client() -> ContainerClient:
    return ContainerClient.from_container_url(SAS_URL)

async def blob_exists(blob_name: str, folder: str) -> bool:
    client: ContainerClient = get_container_client()
    blob_path = f"{folder}/{blob_name}.webp"
    try:
        async with client:
            blob_client = client.get_blob_client(blob_path)
            return await blob_client.exists()
    except Exception:
        return False

def get_cover_and_image_node_ids(frame):
    """
    Return (frame_id, inner_image_rectangle_id) for lesson_cover or lesson_subpart_cover frames.
    """
    def find_image_rect(node):
        if node.get("type") == "RECTANGLE":
            for fill in node.get("fills", []):
                if fill.get("type") == "IMAGE":
                    return node.get("id")
        children = node.get("children")
        if isinstance(children, list):
            for child in children:
                found = find_image_rect(child)
                if found:
                    return found
        return None

    if frame.get("type") != "FRAME":
        return None

    if frame.get("name") in {"lesson_cover"}:
        image_node_id = find_image_rect(frame)
        if image_node_id:
            return (frame["id"], image_node_id)
    return None

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

async def upload_lesson_cover_images_from_structure(
    figma_json_path: str,
    file_id: str,
    module_structure_path: str,
    known_page_names: list[str],
    overwrite: bool = False
):
    with open(module_structure_path, "r", encoding="utf-8") as f:
        structure = json.load(f)

    with open(figma_json_path, "r", encoding="utf-8") as f:
        figma_data = json.load(f)
    figma_children = figma_data.get("document", {}).get("children", figma_data.get("children", []))

    pages_in_figma = {p.get("name"): p for p in figma_children if p.get("type") == "CANVAS"}
    lesson_to_node = {}

    for page_name in known_page_names:
        figma_page = pages_in_figma.get(page_name)
        if not figma_page:
            print(f"⚠️ Skipping: Page '{page_name}' not found.")
            continue

        module_id_match = re.search(r"Module (\d+)", page_name)
        if not module_id_match:
            print(f"⚠️ Skipping: Could not extract module ID from '{page_name}'.")
            continue
        module_id = module_id_match.group(1)

        module = next((m for m in structure["modules"] if str(m["id"]) == module_id), None)
        if not module:
            print(f"⚠️ Skipping: Module {module_id} not found.")
            continue

        lessons_flat = {
            l["id"]: l
            for c in module.get("chapters", [])
            for l in c.get("lessons", [])
            if not l["id"].endswith(".0") and not l["id"].endswith(".-1")
        }

        for section in figma_page.get("children", []):
            if section.get("type") != "SECTION":
                continue

            chapter_match = re.search(r"Chapter (\d+)", section.get("name", ""))
            if not chapter_match:
                continue
            chapter_num = chapter_match.group(1)

            for subsection in section.get("children", []):
                if subsection.get("type") != "SECTION":
                    continue

                sub_name = subsection.get("name", "")
                match = re.search(r"Lesson (\d+)", sub_name)
                if not match:
                    continue
                lesson_num = match.group(1)
                lesson_id = f"{module_id}.{chapter_num}.{lesson_num}"

                if lesson_id not in lessons_flat:
                    continue

                frames = [f for f in subsection.get("children", []) if f.get("type") == "FRAME"]
                for f in reversed(frames):
                    cover_data = get_cover_and_image_node_ids(f)
                    if cover_data:
                        frame_id, image_id = cover_data
                        lesson_to_node[lesson_id] = {"frame": frame_id, "image": image_id}
                        print(f"✅ Matched lesson {lesson_id} with frame and image")
                        break
                else:
                    print(f"⚠️ No lesson_cover found for {lesson_id}")

    print(f"\n🧠 Found {len(lesson_to_node)} lessons to process")

    # Download image URLs
    all_node_ids = []
    for node_pair in lesson_to_node.values():
        all_node_ids.extend([node_pair["frame"], node_pair["image"]])

    headers = {"X-Figma-Token": FIGMA_TOKEN}
    image_urls = {}

    for batch in chunked(all_node_ids, 20):
        res = requests.get(
            f"https://api.figma.com/v1/images/{file_id}",
            params={"ids": ",".join(batch), "format": "png"},
            headers=headers,
            timeout=60,
        )
        res.raise_for_status()
        image_urls.update(res.json().get("images", {}))

    # Upload to Azure
    for lesson_id, nodes in lesson_to_node.items():
        for kind in ["frame", "image"]:
            node_id = nodes[kind]
            blob_name = f"lesson{lesson_id.replace('.', '')}_{kind}"

            try:
                url = image_urls.get(node_id)
                if not url:
                    raise ValueError("No URL returned for node")

                if not overwrite:
                    already_exists = await blob_exists(blob_name, "Lessons"+"/"+kind)
                    if already_exists:
                        print(f"⏭️ Skipped {blob_name} (already exists)")
                        continue

                img_res = requests.get(url)
                img_res.raise_for_status()
                base64_img = base64.b64encode(img_res.content).decode("utf-8")
                webp_data = convert_to_thumbnail(base64_img, resize_to=1080, target_format="webp")
                blob_url = await upload_image_to_azure(
                    blob_name,
                    "Lessons"+"/"+kind,
                    webp_data,
                    overwrite
                )
                print(f"✅ Uploaded {blob_name} → {blob_url}")
            except Exception as e:
                print(f"❌ Failed to process {blob_name}: {e}")


In [5]:
# Use this to start upload:
await upload_lesson_cover_images_from_structure(
    figma_json_path="../02_Inputs/figma_jsons/figma_document1.json",
    file_id=FILE_ID_1,
    module_structure_path="../03_Outputs/SEA_Modules/en/module_structure.json",
    known_page_names=pages_1,
    overwrite=True
)

await upload_lesson_cover_images_from_structure(
    figma_json_path="../02_Inputs/figma_jsons/figma_document2.json",
    file_id=FILE_ID_2,
    module_structure_path="../03_Outputs/SEA_Modules/en/module_structure.json",
    known_page_names=pages_2,
    overwrite=True
)

✅ Matched lesson 1.4.3 with frame and image
✅ Matched lesson 1.4.2 with frame and image
✅ Matched lesson 1.4.1 with frame and image
✅ Matched lesson 1.3.5 with frame and image
✅ Matched lesson 1.3.4 with frame and image
✅ Matched lesson 1.3.3 with frame and image
✅ Matched lesson 1.3.2 with frame and image
✅ Matched lesson 1.3.1 with frame and image
✅ Matched lesson 1.2.3 with frame and image
✅ Matched lesson 1.2.2 with frame and image
✅ Matched lesson 1.2.1 with frame and image
✅ Matched lesson 1.1.3 with frame and image
✅ Matched lesson 1.1.2 with frame and image
✅ Matched lesson 1.1.1 with frame and image
✅ Matched lesson 2.3.5 with frame and image
✅ Matched lesson 2.3.4 with frame and image
✅ Matched lesson 2.3.3 with frame and image
✅ Matched lesson 2.3.2 with frame and image
✅ Matched lesson 2.3.1 with frame and image
✅ Matched lesson 2.2.2 with frame and image
✅ Matched lesson 2.2.3 with frame and image
✅ Matched lesson 2.2.1 with frame and image
✅ Matched lesson 2.1.3 with fram

In [6]:
import os
import base64
import requests
import asyncio
from itertools import islice
from dotenv import load_dotenv
from PIL import Image
import io
from azure.storage.blob import ContentSettings
from azure.storage.blob.aio import ContainerClient
from azure.core.pipeline.transport import AioHttpTransport

# === Load Environment ===
load_dotenv(".env")
FIGMA_TOKEN = os.getenv("FIGMA_TOKEN")
INFOGRAPHIC_FIGMA_DOCUMENT_ID = os.getenv("INFOGRAPHIC_FIGMA_DOCUMENT_ID")
SAS_URL = os.getenv("SAS_URL")
USE_INSECURE_SSL = False

MODULE_PAGES = [f"Module {i}" for i in range(1, 9)]

# === Utility Functions ===

def convert_to_webp(base64_str, resize_to=1080):
    img_bytes = base64.b64decode(base64_str)
    with Image.open(io.BytesIO(img_bytes)) as img:
        img = img.convert("RGB")
        if img.width > resize_to or img.height > resize_to:
            img.thumbnail((resize_to, resize_to), Image.Resampling.LANCZOS)
        out = io.BytesIO()
        img.save(out, format="WEBP", quality=95)
        return out.getvalue()

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

async def upload_blob(folder: str, name: str, data: bytes):
    path = f"{folder}/{name}.webp"
    if USE_INSECURE_SSL:
        import ssl
        ssl_context = ssl.create_default_context()
        ssl_context.check_hostname = False
        ssl_context.verify_mode = ssl.CERT_NONE
        transport = AioHttpTransport(ssl_context=ssl_context)
        client = ContainerClient.from_container_url(SAS_URL, transport=transport)
    else:
        client = ContainerClient.from_container_url(SAS_URL)

    async with client:
        blob_client = client.get_blob_client(path)
        await blob_client.upload_blob(
            data,
            overwrite=True,
            content_settings=ContentSettings(content_type="image/webp"),
        )
        print(f"✅ Uploaded {path}")

def get_figma_file(file_id: str) -> dict:
    res = requests.get(
        f"https://api.figma.com/v1/files/{file_id}",
        headers={"X-Figma-Token": FIGMA_TOKEN}
    )
    res.raise_for_status()
    return res.json()

def get_section_frames(page_node):
    """
    Collect frames that are direct children of top-level sections within the page.
    Returns list of (frame_node, section_name)
    """
    frames = []
    for section in page_node.get("children", []):
        if section.get("type") != "SECTION":
            continue
        section_name = section.get("name", "Unnamed_Section").replace(" ", "_")
        for child in section.get("children", []):
            if child.get("type") == "FRAME":
                frames.append((child, section_name))
    return frames

# === Main Worker ===

async def export_module_frames(file_id: str, module_pages: list[str]):
    data = get_figma_file(file_id)
    canvases = {p["name"]: p for p in data["document"]["children"] if p.get("type") == "CANVAS"}
    headers = {"X-Figma-Token": FIGMA_TOKEN}

    for module in module_pages:
        print(f"\n📦 Processing {module}")
        page = canvases.get(module)
        if not page:
            print(f"⚠️ Page '{module}' not found.")
            continue

        frame_entries = get_section_frames(page)
        if not frame_entries:
            print(f"⚠️ No section-contained frames found in {module}")
            continue

        node_ids = [frame["id"] for frame, _ in frame_entries]
        node_names = {
            frame["id"]: f"{frame['name'].replace(' ', '_')}"
            for frame, section in frame_entries
        }

        for batch in chunked(node_ids, 20):
            res = requests.get(
                f"https://api.figma.com/v1/images/{file_id}",
                headers=headers,
                params={"ids": ",".join(batch), "format": "png"},
                timeout=60
            )
            res.raise_for_status()
            urls = res.json().get("images", {})

            for node_id in batch:
                try:
                    url = urls.get(node_id)
                    if not url:
                        print(f"⚠️ No image for {node_id}")
                        continue

                    response = requests.get(url)
                    response.raise_for_status()

                    img_data = base64.b64encode(response.content).decode("utf-8")
                    webp_bytes = convert_to_webp(img_data)
                    filename = node_names.get(node_id, node_id.replace(":", "_"))
                    folder = f"Modules/{module.replace(' ', '_')}"

                    await upload_blob(folder, filename, webp_bytes)
                except Exception as e:
                    print(f"❌ Error processing frame {node_id}: {e}")

# === Entrypoint ===

await export_module_frames(INFOGRAPHIC_FIGMA_DOCUMENT_ID, MODULE_PAGES)



📦 Processing Module 1
✅ Uploaded Modules/Module_1/chart.webp
✅ Uploaded Modules/Module_1/M1_C1_1.webp
✅ Uploaded Modules/Module_1/M1_C1_3.webp
✅ Uploaded Modules/Module_1/M1_C1_4.webp
✅ Uploaded Modules/Module_1/chart.webp
✅ Uploaded Modules/Module_1/chart.webp
✅ Uploaded Modules/Module_1/M1_C1_A.webp
✅ Uploaded Modules/Module_1/chart.webp
✅ Uploaded Modules/Module_1/M1_C1_6.webp
✅ Uploaded Modules/Module_1/M1_C1_6.webp
✅ Uploaded Modules/Module_1/M1_C1_7.webp
✅ Uploaded Modules/Module_1/M1_C1_8.webp
✅ Uploaded Modules/Module_1/chart.webp
✅ Uploaded Modules/Module_1/M1_C1_C.webp
✅ Uploaded Modules/Module_1/chart.webp
✅ Uploaded Modules/Module_1/chart.webp
✅ Uploaded Modules/Module_1/chart.webp
✅ Uploaded Modules/Module_1/M3_C2_5.webp
✅ Uploaded Modules/Module_1/M1_C3_6.webp
✅ Uploaded Modules/Module_1/M1_C6_14.webp
✅ Uploaded Modules/Module_1/M1_C6_15.webp
✅ Uploaded Modules/Module_1/M1_C6_16.webp
✅ Uploaded Modules/Module_1/M1_C6_17.webp
✅ Uploaded Modules/Module_1/M1_C6_18.webp
✅ Up